In [1]:
from pathlib import Path
import os
dirs_to_do = []
for dirloop in os.listdir('rsr_stats'):
    the_dir = 'rsr_stats/' + dirloop
    lastmod = None
    jsonmod = None
    for fname in os.listdir(the_dir):
        mtimestamp = Path(the_dir + '/' + fname).stat().st_mtime
        if fname.endswith('.json'):
            jsonmod = mtimestamp
        else:
            if not lastmod:
                lastmod = mtimestamp
            else:
                lastmod = max(mtimestamp, lastmod)
    if lastmod > jsonmod:
        dirs_to_do.append(the_dir)
print(dirs_to_do)

[]


In [23]:
from bs4 import BeautifulSoup
from pyjsparser import parse
from pprint import pprint
import json
import statistics
from scipy import stats
import numpy
import os


def convert_str2time(strg):
    m, s = strg.split(':')
    return int(m) * 60 + float(s)


def get_laptimes_and_counts(filename):
    with open(filename) as f:
        content = f.read()
    soup = BeautifulSoup(content, 'html.parser')
    all_js = soup.find_all('script', attrs={'src':None})
    js = ''.join(str(all_js[1]).splitlines(keepends=True)[1:-1])
    
    # USE print statement below to copy/paste into vim as a .json file, and open this json file with a browser
    # print(json.dumps(parsed))
    
    parsed = parse(js)['body']

    part1 = parsed[0]['expression']['arguments'][0]['body']['body'][2]['expression']['arguments'][0]['properties']
    part2 = part1[2]['value']['properties'][0]['value']['elements']
    laptimes = [convert_str2time(x['value']) for x in part2]
    part2 = part1[6]['value']['elements'][0]['properties'][1]['value']['elements']
    counts = [int(x['value']) for x in part2]
    assert len(counts) == len(laptimes)
    return (laptimes, counts)

def get_lapcarvalue(laptimes, counts):
    """gimme the default laptime for a car on the track."""
    lst = []
    for l in laptimes:
        for c in counts:
            lst.append(l)
    return numpy.percentile(lst, 40)

def create_track_json(trackdir):
    """a given directory for a track - e.g. 'rsr_stats/2'.
    save the json file for the cars like
    {'CARID': SECONDS, 'CARID': SECONDS,...}"""
    
    result = {}
    for plainfilename in os.listdir(trackdir):
        if plainfilename.endswith('.html'):
            carid = plainfilename.split('.')[0]
            fname = trackdir + '/' + plainfilename
            laptimes, counts = get_laptimes_and_counts(fname)
            if sum(counts) > 24:
                result[carid] = get_lapcarvalue(laptimes, counts)
                # print("done car {0} with {1} entries".format(carid, sum(counts)))
            else:
                print("only {0} times for car{1}. skipping.".format(sum(counts), carid))
    with open(trackdir + '/times.json', 'w') as fp:
        fp.write(json.dumps(result))
        print('############### {0} done.'.format(trackdir))
    return

for d in dirs_to_do:
    create_track_json(d)
    